# 🧠 Entropy-Driven CfC Context Pruning

Bu notebook, tüm deney pipeline'ını uçtan uca çalıştırır.
- `SMOKE_TEST = True`  → T4 üzerinde hızlı duman testi (~birkaç dakika)
- `SMOKE_TEST = False` → A100 üzerinde tam deney

In [ ]:
# ============================================================
# SMOKE TEST FLAG xx –  T4 için True, A100 tam koşum için False
# ============================================================
SMOKE_TEST = True
SMOKE_FLAG = "--smoke_test" if SMOKE_TEST else ""

## 1. Ortam Kurulumu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_dir = '/content/drive/MyDrive/CENG_467'
if not os.path.exists(drive_dir):
    os.makedirs(drive_dir)
    print(f"✓ Klasör oluşturuldu: {drive_dir}")
else:
    print(f"✓ Drive bağlandı: {drive_dir}")

In [ ]:
import os
REPO_DIR = '/content/CENG467_Final'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Mrtuzy/CENG467_Final.git $REPO_DIR
else:
    print(f'{REPO_DIR} zaten mevcut, git pull yapılıyor...')
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!pwd

# HF_TOKEN hâlâ set mi kontrol et (restart sonrası silinir)
if not os.environ.get("HF_TOKEN"):
    print("⚠️  HF_TOKEN bulunamadı — Hücre 5'i tekrar çalıştır!")

In [ ]:
from google.colab import userdata
import os

token = userdata.get("HF_TOKEN")
if not token:
    raise ValueError("HF_TOKEN bulunamadi. Colab Secrets > 'HF_TOKEN' ekli oldugundan emin ol.")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_TOKEN"] = token
print("✓ HF_TOKEN ayarlandı.")

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate bitsandbytes

# ⚠️  KURULUM SONRASI ZORUNLU ADIM:
# Runtime > Restart session  (veya Ctrl+M .)
#
# Restart sonrası şu hücreleri TEKRAR çalıştır (sırayla):
#   1. Hücre 1  — SMOKE_TEST flag
#   2. Hücre 3  — Drive mount + klasör doğrulama
#   3. Hücre 4  — git clone/pull  +  %cd
#   4. Hücre 5  — HF_TOKEN
# Ardından buradan (Veri Hazırlığı, Adım 2) devam et.
print("Kurulum tamamlandı. Şimdi Runtime > Restart session yapın.")

In [ ]:
# HuggingFace Token (Mistral-7B erişimi için)
# Yöntem A: Colab Secrets'a 'HF_TOKEN' ekleyin
# Yöntem B: Aşağıdaki satırı düzenleyin:
# import os; os.environ['HF_TOKEN'] = 'hf_xxx'

## 2. Veri Hazırlığı (QReCC)

In [ ]:
!python src/data_prep.py $SMOKE_FLAG

## 3. Öğretmen Etiketleme (Mistral-7B, Leave-One-Out)

In [ ]:
!python src/teacher_labeling.py $SMOKE_FLAG

# GPU belleği temizle
import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 4. SBERT Vektörizasyon + DistilGPT-2 Entropi → Δt

In [ ]:
!python src/build_inputs.py $SMOKE_FLAG

import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 5. CfC Ağı Eğitimi

In [ ]:
!python src/train_cfc.py $SMOKE_FLAG

In [ ]:
# Training loss grafiğini göster
from IPython.display import Image, display
import os
fig_path = '/content/drive/MyDrive/CENG_467/figures/training_loss.png'
if os.path.exists(fig_path):
    display(Image(filename=fig_path, width=600))
else:
    print('Training loss grafiği bulunamadı.')

## 6. Değerlendirme (Full / CfC / Random / Cosine)

In [ ]:
!python src/evaluate.py $SMOKE_FLAG

import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 7. Sonuçlar ve Grafikler

In [ ]:
import json, os
results_path = '/content/drive/MyDrive/CENG_467/outputs/eval_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

    # Tüm mevcut metrikleri dinamik al
    all_metrics = list(next(iter(results.values())).keys())
    col_w = 12
    header = f"{'Method':<{col_w}}" + "".join(f"{m:>{col_w}}" for m in all_metrics)
    print(header)
    print('-' * len(header))
    for method, vals in results.items():
        row = f"{method:<{col_w}}"
        for m in all_metrics:
            v = vals.get(m, '-')
            row += f"{v:>{col_w}}" if isinstance(v, str) else f"{v:>{col_w}.4f}"
        print(row)
else:
    print('Sonuç dosyası bulunamadı.')

In [ ]:
# Tüm grafikleri göster
from IPython.display import Image, display
import glob

fig_dir = '/content/drive/MyDrive/CENG_467/figures'
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))

if figs:
    for fp in figs:
        print(f'\n📊 {os.path.basename(fp)}')
        display(Image(filename=fp, width=600))
else:
    print('Grafik bulunamadı.')

## 8. Ablation Study – TAU Eşiği Duyarlılığı

TAU (pruning threshold) değerini 0.2–0.8 arasında sweep ederek CfC'nin kalite/verimlilik dengesini ölçer.

In [ ]:
!python src/ablation.py $SMOKE_FLAG

import torch; torch.cuda.empty_cache()
import gc; gc.collect()

In [ ]:
# Ablation sonuçlarını tablo + grafik olarak göster
import json, os
from IPython.display import Image, display

abl_path = '/content/drive/MyDrive/CENG_467/outputs/ablation_results.json'
if os.path.exists(abl_path):
    with open(abl_path) as f:
        abl = json.load(f)
    print(f"{'TAU':<8} {'ROUGE-L':>10} {'Avg_Tokens':>12} {'Reduction%':>12}")
    print('-' * 46)
    for tau, vals in sorted(abl.items(), key=lambda x: float(x[0])):
        print(f"{float(tau):<8.2f} {vals['ROUGE-L']:>10.4f}"
              f" {vals['Avg_Tokens']:>12.1f} {vals['Reduction%']:>11.1f}%")
else:
    print('Ablasyon sonucu bulunamadı.')

fig_path = '/content/drive/MyDrive/CENG_467/figures/ablation_tau_sweep.png'
if os.path.exists(fig_path):
    display(Image(filename=fig_path, width=650))

## 9. Qualitative Error Analysis

Her metod için en iyi ve en kötü 5 tahmin örneğini gösterir.

In [ ]:
import json, os

qual_path = '/content/drive/MyDrive/CENG_467/outputs/qualitative_examples.json'
if not os.path.exists(qual_path):
    print('Qualitative örnekler bulunamadı. Önce evaluate.py çalıştırın.')
else:
    with open(qual_path, encoding='utf-8') as f:
        qual = json.load(f)

    SHOW_METHODS = ['full', 'cfc', 'random', 'cosine']
    SHOW_N = 3  # her kategoriden kaç örnek gösterilsin

    for method in SHOW_METHODS:
        if method not in qual:
            continue
        print(f"\n{'='*70}")
        print(f"  METHOD: {method.upper()}")
        print(f"{'='*70}")
        for category, label in [('worst', '❌ WORST'), ('best', '✅ BEST')]:
            print(f"\n  {label} (düşük → yüksek ROUGE-L)\n")
            for i, ex in enumerate(qual[method][category][:SHOW_N]):
                print(f"  [{i+1}] ROUGE-L = {ex['rougeL']:.4f}")
                print(f"       Q:    {ex['question'][:120]}")
                print(f"       REF:  {ex['reference'][:120]}")
                print(f"       PRED: {ex['prediction'][:120]}")
                if method == 'cfc' and ex.get('cfc_kept'):
                    kept_turns = ex['cfc_kept'].split(' | ')
                    print(f"       KEPT ({len(kept_turns)} turn): {ex['cfc_kept'][:150]}")
                print()

---
### ✅ Deney tamamlandı!

Tüm sonuçlar ve grafikler Google Drive'da:
- `MyDrive/CENG_467/outputs/eval_results.json`
- `MyDrive/CENG_467/figures/*.png`
- `MyDrive/CENG_467/models/best_cfc_model.pth`